# 🎮 Video Game Sales Prediction using Machine Learning

## Predicción de Ventas de Videojuegos mediante Machine Learning

### Business Understanding / Comprensión del Negocio

**English**

The objective of this project is to predict global video game sales using information available before the game is released. Different regression models are compared in order to identify the most suitable approach for this problem.

**Español**

El objetivo de este proyecto es predecir las ventas globales de videojuegos utilizando únicamente información disponible antes del lanzamiento del juego. Se comparan distintos modelos de regresión para identificar el enfoque con mejor desempeño.


# 1. Import Libraries / Importación de Librerías

The following libraries are required for data manipulation, visualization and Machine Learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2. Load Dataset / Carga del Dataset

The dataset is loaded into a Pandas DataFrame and inspected before preprocessing begins.

In [ ]:
games_df = pd.read_csv("Data_sets\\Videojuegos\\vgchartz-2024.csv")
games_df.head()

# 3. Data Understanding / Comprensión de los Datos

The structure of the dataset is inspected, including dimensions, data types, duplicated rows and missing values.

In [ ]:
print(games_df.shape)
games_df.info()
print("Duplicated rows:", games_df.duplicated().sum())

# 4. Data Cleaning / Limpieza de Datos

**Removed features**

- Data leakage (`na_sales`, `jp_sales`, `pal_sales`, `other_sales`)
- Irrelevant information (`img`, `title`)
- Information unavailable before release (`critic_score`, `last_update`)

Rows without the target variable are removed because supervised learning requires known labels.


In [ ]:
games_df = games_df.dropna(subset=["total_sales"])

games_df = games_df.drop(columns=[
    "img","title","critic_score",
    "na_sales","jp_sales","pal_sales","other_sales",
    "last_update"
])

games_df = games_df.dropna(subset=["developer","release_date"])

games_df.info()

# 5. Feature Engineering / Ingeniería de Variables

The release date is converted into datetime format and transformed into two numerical features:

- release_year
- release_month


In [ ]:
games_df["release_date"] = pd.to_datetime(games_df["release_date"])

games_df["release_year"] = games_df["release_date"].dt.year
games_df["release_month"] = games_df["release_date"].dt.month

games_df = games_df.drop(columns=["release_date"])

# 6. Categorical Encoding / Codificación de Variables

Two encoding strategies are used.

| Feature | Encoding |
|---------|----------|
| publisher | Frequency Encoding |
| developer | Frequency Encoding |
| console | One-Hot Encoding |
| genre | One-Hot Encoding |


In [ ]:
publisher_frequency = games_df["publisher"].value_counts(normalize=True)
games_df["publisher"] = games_df["publisher"].map(publisher_frequency)

developer_frequency = games_df["developer"].value_counts(normalize=True)
games_df["developer"] = games_df["developer"].map(developer_frequency)

games_df = pd.get_dummies(
    games_df,
    columns=["console","genre"],
    dtype=int
)

games_df.shape

# 7. Exploratory Data Analysis (EDA)

The target variable is analyzed to identify skewness and justify the models evaluated later.


In [ ]:
print(games_df["total_sales"].describe())

plt.figure(figsize=(8,5))
plt.hist(games_df["total_sales"], bins=50)
plt.title("Distribution of Total Sales")
plt.xlabel("Total Sales (Millions)")
plt.ylabel("Frequency")
plt.show()

# 8. Train/Test Split

The dataset is divided into training (80%) and testing (20%) subsets.


In [ ]:
X = games_df.drop(columns=["total_sales"])
y = games_df["total_sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 9. Model 1 — Linear Regression

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train,y_train)

y_pred = linear_model.predict(X_test)

print("Linear Regression")
print("MAE :",mean_absolute_error(y_test,y_pred))
print("RMSE:",np.sqrt(mean_squared_error(y_test,y_pred)))
print("R²  :",r2_score(y_test,y_pred))

# 10. Model 2 — Linear Regression (Log Target)

In [ ]:
y_train_log = np.log1p(y_train)

linear_model_log = LinearRegression()
linear_model_log.fit(X_train,y_train_log)

y_pred = np.expm1(linear_model_log.predict(X_test))

print("Linear Regression + Log")
print("MAE :",mean_absolute_error(y_test,y_pred))
print("RMSE:",np.sqrt(mean_squared_error(y_test,y_pred)))
print("R²  :",r2_score(y_test,y_pred))

# 11. Model 3 — Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train,y_train)

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test,y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test,y_pred_rf))
r2_rf = r2_score(y_test,y_pred_rf)

print("Random Forest")
print("MAE :",mae_rf)
print("RMSE:",rmse_rf)
print("R²  :",r2_rf)

# 12. Feature Importance

Random Forest feature importance is used to identify the variables that contributed the most to the predictions.


In [ ]:
importance_df = pd.DataFrame({
    "Feature":X.columns,
    "Importance":rf_model.feature_importances_
}).sort_values("Importance",ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(10,6))
plt.barh(
    importance_df["Feature"].head(15),
    importance_df["Importance"].head(15)
)
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Top 15 Most Important Features")
plt.show()

# 13. Conclusions / Conclusiones

## Results

| Model | Summary |
|------|---------|
| Linear Regression | Baseline model with limited predictive capability. |
| Linear Regression + Log | Reduced MAE but did not improve overall performance. |
| Random Forest | Best overall model, achieving the highest R² and the lowest prediction errors. |

## Main Findings

- `release_year` was the most influential feature.
- `developer` and `publisher` contributed significantly to prediction performance.
- Random Forest captured nonlinear relationships better than Linear Regression.

## Future Work

- Hyperparameter tuning.
- Cross-validation.
- Target Encoding.
- Gradient Boosting (XGBoost, LightGBM or CatBoost).
